Set up Code- use for both agents 

In [1]:
#Packages Used
import os
import json
import re
import requests
import pandas as pd

from pathlib import Path
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#Model Information 
CLIENT_ID = ""
CLIENT_SECRET = ""
API_KEY = ""
TOKEN_URL = ""
SCOPE = ""
CHAT_URL = ""
MODEL = ""

#Agent Instructions 
AGENT_INSTRUCTIONS = """
You are an agent meant to grade answers from a research study. You will be given a file containing the participant ID, the question, and the participant’s answer. You will grade the answers based only on your past knowledge. 

You will add a column to the file called AI Grade and grade each answer as either 1 (correct) or 0 (incorrect). Grade every line with either 1 or 0. 

You have also been provided with a sample of questions and grades. Use these examples as a guideline for how to grade the responses. 
"""

#Knowledge Files
KNOWLEDGE_FILES = [
    "AI Train Data 1.xlsx"
]

Code for Gemini Agents 

In [ ]:

#Bulding Agents
class GeminiResearchAgent:

    def __init__(
        self,
        name,
        knowledge_base,
        instructions,
        model=GEMINI_MODEL,
    ):
        self.name = name
        self.knowledge_base = knowledge_base
        self.instructions = instructions
        self.model = model
        self.history = []

        self.training_rows = []

        for file_path in self.knowledge_base:

            sheets = pd.read_excel(
                file_path,
                sheet_name=None
            )

            for sheet_name, df in sheets.items():

                for _, row in df.iterrows():

                    row_text = " | ".join(
                        f"{column}: {row[column]}"
                        for column in df.columns
                    )

                    self.training_rows.append(row_text)
        self.vectorizer = TfidfVectorizer()
        self.training_vectors = self.vectorizer.fit_transform(
            self.training_rows
        )


    def ask(self, question, top_k=5):
        question_vector = self.vectorizer.transform(
            [question]
        )

        similarities = cosine_similarity(
            question_vector,
            self.training_vectors
        )[0]


        top_indices = similarities.argsort()[-top_k:][::-1]

        knowledge_text = "\n\n".join(
            self.training_rows[i]
            for i in top_indices
        )

        token_response = requests.post(
            TOKEN_URL,
            timeout=30,
            data={
                "grant_type": "client_credentials",
                "client_id": CLIENT_ID,
                "client_secret": CLIENT_SECRET,
                "scope": SCOPE,
            },
        )

        token_response.raise_for_status()

        token = token_response.json()["access_token"]


        headers = {
            "Authorization": f"Bearer {token}",
            "X-Api-Key": API_KEY,
            "Content-Type": "application/json",
        }

        response = requests.post(
            CHAT_URL,
            headers=headers,
            timeout=120,
            json={
                "model": self.model,
                "contents": [
                    {
                        "role": "user",
                        "parts": [
                            {
                                "text": f"""
INSTRUCTIONS
============
{self.instructions}

RELEVANT TRAINING EXAMPLES
==========================
{knowledge_text}

RESPONSE TO GRADE
=================
{question}
"""
                            }
                        ]
                    }
                ]
            },
        )

        response.raise_for_status()

        data = response.json()

        answer = (
            data["candidates"][0]
            ["content"]
            ["parts"][0]
            ["text"]
        )

        usage = data.get("usageMetadata", {})

        return {
            "agent": self.name,
            "model": self.model,
            "question": question,
            "answer": answer,
            "prompt_tokens": usage.get("promptTokenCount", 0),
            "output_tokens": usage.get("candidatesTokenCount", 0),
            "total_tokens": usage.get("totalTokenCount", 0)
        }

In [ ]:
#Running Agent
test_file = "AI Test Data.xlsx"
test_df = pd.read_excel(test_file)

grades = []
running_tokens = 0

for index, row in test_df.iterrows():

    participant_id = row.iloc[0]
    question = row.iloc[1]
    participant_answer = row.iloc[2]

    print(f"Starting Row {index + 1}...", flush=True)

#Put Prompt Here
    grading_prompt = f"""
Grade the participant response using the training examples
and grading knowledge you were provided.

Question:
{question}

Participant Response:
{participant_answer}

Return ONLY:
1 if the answer is correct
0 if the answer is incorrect
"""

    try:
        result = agent1.ask(
            grading_prompt,
            top_k=5
        )

        grade = result["answer"].strip()

        running_tokens += result["total_tokens"]

        print(
            f"Row {index + 1}: "
            f"Participant {participant_id} -> {grade} | "
            f"Tokens: {result['total_tokens']} | "
            f"Running Total: {running_tokens}",
            flush=True
        )

        if grade not in ["0", "1"]:
            print(
                f"WARNING: Unexpected grade for row {index + 1}: {grade}"
            )

        grades.append(grade)

    except Exception as e:
        print(
            f"ERROR on Row {index + 1}: {e}",
            flush=True
        )
        break

Code for ChatGPT Agents 

In [ ]:
class GPTResearchAgent:

    def __init__(
        self,
        name,
        knowledge_base,
        instructions,
        model=GPT_MODEL,
    ):
        self.name = name
        self.knowledge_base = knowledge_base
        self.instructions = instructions
        self.model = model
        self.history = []

        self.training_rows = []

        for file_path in self.knowledge_base:

            sheets = pd.read_excel(
                file_path,
                sheet_name=None
            )

            for sheet_name, df in sheets.items():

                for _, row in df.iterrows():

                    row_text = " | ".join(
                        f"{column}: {row[column]}"
                        for column in df.columns
                    )

                    self.training_rows.append(row_text)

        self.vectorizer = TfidfVectorizer()

        self.training_vectors = self.vectorizer.fit_transform(
            self.training_rows
        )


    def ask(self, question, top_k=5):

        question_vector = self.vectorizer.transform(
            [question]
        )

        similarities = cosine_similarity(
            question_vector,
            self.training_vectors
        )[0]

        top_indices = similarities.argsort()[-top_k:][::-1]

        knowledge_text = "\n\n".join(
            self.training_rows[i]
            for i in top_indices
        )
        token_response = requests.post(
            TOKEN_URL,
            timeout=30,
            data={
                "grant_type": "client_credentials",
                "client_id": CLIENT_ID,
                "client_secret": CLIENT_SECRET,
                "scope": SCOPE,
            },
        )

        token_response.raise_for_status()

        token = token_response.json()["access_token"]

        headers = {
            "Authorization": f"Bearer {token}",
            "X-Api-Key": API_KEY,
            "Content-Type": "application/json",
        }

        response = requests.post(
            CHAT_URL,
            headers=headers,
            timeout=120,
            json={
                "model": self.model,
                "messages": [
                    {
                        "role": "system",
                        "content": self.instructions
                    },
                    {
                        "role": "user",
                        "content": f"""
RELEVANT TRAINING EXAMPLES
==========================
{knowledge_text}

RESPONSE TO GRADE
=================
{question}
"""
                    }
                ]
            },
        )

        response.raise_for_status()

        data = response.json()

        answer = (
            data["choices"][0]
            ["message"]
            ["content"]
        )

        usage = data.get("usage", {})

        return {
            "agent": self.name,
            "model": self.model,
            "question": question,
            "answer": answer,
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "output_tokens": usage.get("completion_tokens", 0),
            "total_tokens": usage.get("total_tokens", 0)
        }

In [ ]:
#Run Agent 
test_file = "AI Test Data 4.xlsx"
test_df = pd.read_excel(test_file)

grades = []
running_tokens = 0

for index, row in test_df.iloc[0:].iterrows():

    participant_id = row.iloc[0]
    question = row.iloc[1]
    participant_answer = row.iloc[2]

    print(f"Starting Row {index + 1}...", flush=True)
#put grading prompt here
    grading_prompt = f"""
Grade the participant response using the training examples
and grading knowledge you were provided.

Question:
{question}

Participant Response:
{participant_answer}

Return ONLY:
1 if the answer is correct
0 if the answer is incorrect
"""

    try:
        result = agent4.ask(
            grading_prompt,
            top_k=5
        )

        grade = result["answer"].strip()

        running_tokens += result["total_tokens"]

        print(
            f"Row {index + 1}: "
            f"Participant {participant_id} -> {grade} | "
            f"Tokens: {result['total_tokens']} | "
            f"Running Total: {running_tokens}",
            flush=True
        )

        if grade not in ["0", "1"]:
            print(
                f"WARNING: Unexpected grade for row {index + 1}: {grade}"
            )

        grades.append(grade)

    except Exception as e:
        print(
            f"ERROR on Row {index + 1}: {e}",
            flush=True
        )
        break